# Beyond Majority Voting: Selecting LLM Answers via Hidden State Trajectory Probes

**Author:** Nikolay Yudin (`n.yudin@gmail.com`)
**Repository:** [github.com/nick-yudin/Generalization/.../Latent\_control](https://github.com/nick-yudin/Generalization/tree/main/papers/Latent_control)

## Abstract

When a language model generates multiple candidate answers, how should we pick the best one? The default strategy — majority voting — treats the model as a black box, discarding everything except final answer strings. We show that the model's internal computations already contain a usable signal for answer quality, and that a remarkably simple method can extract it.

We propose *trajectory probes*: lightweight linear models trained on hidden-state features aggregated across the generation process. From each candidate answer, we extract mean, standard deviation, and final-token activations at eight evenly spaced layers, projected to 256 dimensions — a 6,144-dimensional trajectory fingerprint. A logistic regression probe trained with a pairwise ranking objective (RankNet) learns to prefer correct answers over incorrect ones from the same question.

On TriviaQA (Llama-3.1-8B-Instruct, *t*=0.3, *n*=500, *K*=4, 3 seeds), the probe improves over majority voting by **+5.1 ± 0.1 pp** (56.4% vs 51.3%), recovering 58% of the gap to the oracle upper bound, with a selection precision (PickAcc) of 91.2 ± 1.7%. On MATH (*n*=500, *K*\_gen=6, 3 seeds), the relationship is *K*-dependent: the probe outperforms MV by +2.1 pp at *K*\_eval=2 (all seeds positive) but the advantage narrows to +0.6 pp at *K*\_eval=4, as MV benefits more from additional votes in mathematical reasoning. The probe trains in under 60 seconds on CPU from 500 examples and adds zero latency at inference.

Two findings surprised us. First, the choice of training objective matters more than feature quality: a binary classifier with higher cross-validated AUC can underperform a pairwise probe with lower AUC, because ranking among candidates is a fundamentally different task than classifying correctness in isolation. Second, the per-layer signal distribution acts as a domain fingerprint — factual recall spreads information across all layers while mathematical reasoning concentrates it in the final third — yet a single probe trained on mixed-domain data matches domain-specific specialists with no interference.

Our results suggest that the "verifier" for best-of-*K* selection need not be a separate model or an additional LLM call. It can be a linear function of what the model already computes.

---
## Notebook 00: Reproduce All Figures from Released Data

**Runtime:** CPU only, ~1 minute
**Input:** `data/paper2_results.json`, `data/paper2_test_ckpts.json`, `data/paper2_layer_aucs.json`
**Output:** All tables and figures from the paper

### Expected outputs
- **Table 1:** Main results (Base / MV / Probe / Oracle × 2 domains × 3 seeds)
- **Figure 1:** Per-layer probe AUC (TriviaQA vs MATH side-by-side)
- **Figure 2:** K-sweep (Probe vs MV accuracy at different K\_eval)
- **Figure 3:** PickAcc vs Recovery scatter (all configurations)
- **Figure 4:** Length confound bars (with / without length features)
- **Figure 5:** Cross-domain transfer heatmap

In [ ]:
import json, os
import numpy as np

# Auto-detect: running on Colab or locally?
if os.path.exists('/content'):
    # Colab: clone repo
    import subprocess
    if not os.path.exists('/content/paper2_release'):
        subprocess.run(['git', 'clone',
                        'https://github.com/nick-yudin/Generalization.git',
                        '/content/repo'], check=True)
        os.symlink('/content/repo/papers/Latent_control', '/content/paper2_release')
    DATA = '/content/paper2_release/data'
else:
    DATA = 'data'

results = json.load(open(f'{DATA}/paper2_results.json'))
test_ckpts = json.load(open(f'{DATA}/paper2_test_ckpts.json'))
print(f'Loaded {len(results)} experiment results, {len(test_ckpts)} test checkpoints')

## Table 1: Main Results (3-seed)

In [ ]:
# ── Table 1: Main Results ──
import sys
sys.path.insert(0, '.' if os.path.exists('paper2_utils.py') else '/content/paper2_release')
from paper2_utils import bootstrap_delta, mcnemar_test

def seed_summary(prefix, seeds=[0, 1, 2]):
    '''Compute mean +/- std across seeds.'''
    keys = ['base_acc', 'mv_acc', 'probe_acc', 'oracle_acc', 'pickacc', 'recovery']
    vals = {k: [] for k in keys}
    for s in seeds:
        r = results.get(f'{prefix}_seed{s}')
        if r is None:
            continue
        for k in keys:
            vals[k].append(r[k])
    out = {}
    for k in keys:
        v = np.array(vals[k])
        out[k] = (v.mean(), v.std()) if len(v) > 0 else (float('nan'), float('nan'))
    # Delta
    deltas = [results[f'{prefix}_seed{s}']['probe_acc'] -
              results[f'{prefix}_seed{s}']['mv_acc']
              for s in seeds if f'{prefix}_seed{s}' in results]
    d = np.array(deltas)
    out['delta'] = (d.mean() * 100, d.std() * 100)
    out['n_seeds'] = len(deltas)
    return out

# Pooled stats from test_ckpts
def pooled_stats(prefix, seeds=[0, 1, 2]):
    all_ckpt = []
    for s in seeds:
        ck = test_ckpts.get(f'{prefix}_seed{s}', [])
        all_ckpt.extend(ck)
    if not all_ckpt:
        return None
    return bootstrap_delta(all_ckpt), mcnemar_test(all_ckpt)

print('='*75)
print('TABLE 1: Main Results')
print('='*75)
print(f'{"Domain":<10} {"Base":>10} {"MV":>10} {"Probe":>10} {"Δ(P−MV)":>12} '
      f'{"Oracle":>10} {"PickAcc":>10} {"Recov":>8}')
print('-'*75)

for prefix, domain in [('v19', 'TriviaQA'), ('v20', 'MATH')]:
    s = seed_summary(prefix)
    if s['n_seeds'] == 0:
        print(f'{domain:<10} (no data yet)')
        continue
    pm = '±'
    print(f'{domain:<10} '
          f'{s["base_acc"][0]:>5.1%}{pm}{s["base_acc"][1]*100:>3.1f} '
          f'{s["mv_acc"][0]:>5.1%}{pm}{s["mv_acc"][1]*100:>3.1f} '
          f'{s["probe_acc"][0]:>5.1%}{pm}{s["probe_acc"][1]*100:>3.1f} '
          f'{s["delta"][0]:>+5.1f}{pm}{s["delta"][1]:>3.1f}pp '
          f'{s["oracle_acc"][0]:>5.1%}{pm}{s["oracle_acc"][1]*100:>3.1f} '
          f'{s["pickacc"][0]:>5.1f}{pm}{s["pickacc"][1]:>3.1f}% '
          f'{s["recovery"][0]:>5.1f}{pm}{s["recovery"][1]:>3.1f}%')

print()
for prefix, domain in [('v19', 'TriviaQA'), ('v20', 'MATH')]:
    ps = pooled_stats(prefix)
    if ps is None:
        continue
    bs, mc = ps
    print(f'{domain}: Pooled Δ = {bs["delta"]*100:+.1f}pp '
          f'CI [{bs["ci_lo"]*100:+.1f}, {bs["ci_hi"]*100:+.1f}] '
          f'McNemar p={mc["p_value"]:.2e} '
          f'(probe {mc["probe_only"]} / MV {mc["mv_only"]})')

## Figure 1: Per-Layer Probe AUC

In [ ]:
import matplotlib.pyplot as plt

try:
    layer_aucs = json.load(open(f'{DATA}/paper2_layer_aucs.json'))
except FileNotFoundError:
    print('No layer AUC data found — skipping Figure 1')
    layer_aucs = {}

if layer_aucs:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
    for ax, (domain, label) in zip(axes, [('trivia', 'TriviaQA'), ('math', 'MATH')]):
        if domain not in layer_aucs:
            ax.set_title(f'{label} (no data)')
            continue
        aucs = layer_aucs[domain]['layer_aucs']
        layers = list(range(len(aucs)))
        ax.bar(layers, aucs, color='steelblue' if domain == 'trivia' else 'coral',
               alpha=0.8)
        ax.axhline(0.5, color='gray', ls='--', lw=0.8)
        ax.set_xlabel('Layer index')
        ax.set_title(label)
        ax.set_ylim(0.45, max(aucs) * 1.05)
    axes[0].set_ylabel('Pairwise probe AUC')
    plt.suptitle('Figure 1: Per-Layer Probe AUC', fontsize=13)
    plt.tight_layout()
    plt.savefig('fig1_layer_auc.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved: fig1_layer_auc.png')

## Figure 2: K-Sweep (Probe vs MV)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, (prefix, domain, k_range) in zip(axes,
    [('v19', 'TriviaQA', [2, 3, 4]), ('v20', 'MATH', [2, 3, 4, 5, 6])]):

    mv_means, probe_means = [], []
    mv_stds, probe_stds = [], []
    valid_ks = []

    for ke in k_range:
        mv_vals, probe_vals = [], []
        for seed in [0, 1, 2]:
            r = results.get(f'{prefix}_seed{seed}')
            if r is None or 'multi_k_eval' not in r:
                continue
            mk = r['multi_k_eval'].get(str(ke))
            if mk:
                mv_vals.append(mk['mv_acc'])
                probe_vals.append(mk['probe_acc'])
        if mv_vals:
            valid_ks.append(ke)
            mv_means.append(np.mean(mv_vals) * 100)
            probe_means.append(np.mean(probe_vals) * 100)
            mv_stds.append(np.std(mv_vals) * 100)
            probe_stds.append(np.std(probe_vals) * 100)

    if not valid_ks:
        ax.set_title(f'{domain} (no multi-K data)')
        continue

    x = np.array(valid_ks)
    ax.errorbar(x - 0.05, mv_means, yerr=mv_stds, marker='s', label='MV',
                capsize=4, color='gray')
    ax.errorbar(x + 0.05, probe_means, yerr=probe_stds, marker='o',
                label='Probe', capsize=4, color='steelblue' if domain == 'TriviaQA' else 'coral')
    ax.set_xlabel('K_eval')
    ax.set_ylabel('Accuracy (%)')
    ax.set_title(domain)
    ax.set_xticks(valid_ks)
    ax.legend()
    ax.grid(alpha=0.3)

plt.suptitle('Figure 2: K-Sweep — Probe vs MV', fontsize=13)
plt.tight_layout()
plt.savefig('fig2_k_sweep.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: fig2_k_sweep.png')

## Figure 3: PickAcc vs Recovery (All Configurations)

In [ ]:
# Scatter all experiments: PickAcc (x) vs Recovery (y)
fig, ax = plt.subplots(figsize=(8, 6))

for key, r in results.items():
    if 'pickacc' not in r or 'recovery' not in r:
        continue
    domain = r.get('domain', '')
    if not domain:
        if 'trivia' in key.lower() or 'trivia' in r.get('version', '').lower():
            domain = 'TriviaQA'
        elif 'math' in key.lower() or 'math' in r.get('version', '').lower():
            domain = 'MATH'
        else:
            domain = 'Other'
    color = 'steelblue' if 'rivia' in domain else ('coral' if 'ATH' in domain else 'gray')
    marker = 'o' if 'rivia' in domain else 's'
    ax.scatter(r['pickacc'], r['recovery'], c=color, marker=marker,
               s=60, alpha=0.7, edgecolors='white', linewidth=0.5)

# Legend
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor='steelblue',
           markersize=8, label='TriviaQA'),
    Line2D([0], [0], marker='s', color='w', markerfacecolor='coral',
           markersize=8, label='MATH'),
]
ax.legend(handles=legend_elements)
ax.set_xlabel('PickAcc (%)')
ax.set_ylabel('Recovery (%)')
ax.set_title('Figure 3: PickAcc vs Recovery (all experiments)')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('fig3_pickacc_recovery.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: fig3_pickacc_recovery.png')

## Figures 4–5: Length Confound & Transfer

*(These require ablation summary data from `paper2_ablation_summary.json`)*

In [ ]:
try:
    ablations = json.load(open(f'{DATA}/paper2_ablation_summary.json'))
except FileNotFoundError:
    print('No ablation summary data — skipping Figures 4–5.')
    ablations = {}

# ── Figure 4: Length Confound ──
if 'length_confound' in ablations:
    lc = ablations['length_confound']
    fig, ax = plt.subplots(figsize=(8, 4))

    labels = ['MV', 'Length\nonly', 'Binary\n(no len)', 'Pairwise\n(no len)',
              'Binary', 'Pairwise']
    keys = ['mv_acc', 'acc_length_only', 'acc_binary_nolen', 'acc_pairwise_nolen',
            'acc_binary', 'acc_pairwise']
    vals = [lc.get(k, 0) for k in keys]
    colors = ['gray', 'khaki', 'lightblue', 'lightcoral', 'steelblue', 'coral']

    # Handle missing keys gracefully
    valid = [(l, v, c) for l, v, c in zip(labels, vals, colors) if v > 0]
    if valid:
        ls, vs, cs = zip(*valid)
        ax.bar(range(len(vs)), [v * 100 for v in vs], color=cs, edgecolor='white')
        ax.set_xticks(range(len(vs)))
        ax.set_xticklabels(ls)
        ax.set_ylabel('Accuracy (%)')
        ax.set_title('Figure 4: Length Confound — MATH')
        ax.grid(alpha=0.3, axis='y')
        plt.tight_layout()
        plt.savefig('fig4_length_confound.png', dpi=150, bbox_inches='tight')
        plt.show()
        print('Saved: fig4_length_confound.png')
    else:
        print('Length confound keys not found in v12 data')

# ── Figure 5: Transfer Heatmap ──
if 'cross_domain' in ablations:
    cd = ablations['cross_domain']
    print('\nCross-domain transfer results:')
    print(json.dumps(cd, indent=2)[:500])
    # Heatmap will depend on exact data structure — placeholder
    print('(Transfer heatmap requires structured train×test matrix — see paper2_02)')

---

**All figures saved as PNG files in the current directory.**

To verify the numbers match the paper, compare Table 1 output above with the paper's Table 1.